# Day 2 — Cleaning
### Real Estate Machine · Graduation Project

Day 1 found the problems. Today we fix them — and **write down why** for every single one.

---

## The rule that governs this whole notebook

> **You may impute a feature. You may never impute a target.**

A missing `bedrooms` value can be estimated from similar houses, and if the estimate is slightly
wrong the model simply gets a slightly noisy input.

A missing `price` is different. `price` is what the model is trying to *learn*. Filling it with a
group average teaches the model to predict group averages, and then your accuracy score measures
how well you reproduced your own guess. That is exactly the mistake baked into `data.csv`.

**Every decision below gets logged**, so that at the end you can show an examiner precisely what
changed and why.

## 1. Setup

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

DATA = Path("..") / "Data"

df = pd.read_csv(DATA / "data_raw_parsed.csv", dtype={"zipcode": str})
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("Loaded:", df.shape)
df.head(3)

Loaded: (4601, 21)


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,address,date_raw,street,city,statezip,country,zipcode
0,2014-05-02,"313,000.00",3.00,1.50,1340,7912,1.50,0,0,3,1340,0,1955,NaN,"18810 Densmore Ave N, Shoreline, WA 98133, USA",20140502T000000,18810 Densmore Ave N,Shoreline,WA 98133,USA,98133
1,2014-05-02,"2,384,000.00",5.00,2.50,3650,9050,2.00,0,4,5,3370,280,1921,NaN,"709 W Blaine St, Seattle, WA 98119, USA",20140502T000000,709 W Blaine St,Seattle,WA 98119,USA,98119
2,2014-05-02,"342,000.00",3.00,2.00,1930,11947,1.00,0,0,4,1930,0,1966,NaN,"26206-26214 143rd Ave SE, Kent, WA 98042, USA",20140502T000000,26206-26214 143rd Ave SE,Kent,WA 98042,USA,98042


### The audit trail

A simple log. Every fix appends one line. At the end we print it as a table — that table goes
straight into your presentation.

In [2]:
cleaning_log = []
start_rows = len(df)

def log(step, action, rows_affected, reason):
    cleaning_log.append({
        "step": step,
        "action": action,
        "rows": rows_affected,
        "reason": reason,
        "rows_remaining": len(df),
    })
    print(f"[{step}] {action}  ->  {rows_affected} rows affected, {len(df)} remaining")

---
## 2. Fix 1 — the duplicate record

One record appears twice, identical in every column. A house cannot be sold twice on the same day
at the same price to produce two identical rows, so this is a data-entry duplication.

Keeping it would let the model see one house twice and slightly overweight it.

In [3]:
dupes = df[df.duplicated(keep=False)]
print("Duplicate rows found:", len(dupes))
dupes[["date", "price", "sqft_living", "street", "city"]]

Duplicate rows found: 2


,date,price,sqft_living,street,city
4336,2014-05-22,"657,500.00",2670,1917 235th Ct NE,Sammamish
4337,2014-05-22,"657,500.00",2670,1917 235th Ct NE,Sammamish


In [4]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
log("Fix 1", "Removed exact duplicate rows", before - len(df),
    "Identical in every column; a single sale recorded twice")

[Fix 1] Removed exact duplicate rows  ->  1 rows affected, 4600 remaining


---
## 3. Fix 2 — misspelled city names

Day 1 found 20 corrupted city names. The naive fix — "replace every city with whatever its
zipcode says" — **is wrong**, and it is worth understanding why before we write any code.

Some zipcodes legitimately contain more than one city, and this dataset contains genuinely small
towns: *Yarrow Point*, *Beaux Arts Village*, *Snoqualmie Pass*, *Preston*, *Skykomish*. A blanket
zipcode override would silently swallow all of them into their larger neighbour — destroying real
information while claiming to clean it.

So we need a rule that is **conservative**: only change a name when we have two independent
reasons to believe it is a typo.

### The rule

A city name is corrected **only if all three conditions hold**:

1. It is **rare** — it appears fewer than 10 times, so it cannot be a real, well-represented town.
2. It is **spelled almost identically** to a frequent name (similarity ≥ 0.75), or differs only in
   capitalisation.
3. That frequent name is the **majority city of this row's own zipcode**.

Condition 3 is what protects the small towns. `Snoqualmie Pass` looks very similar to
`Snoqualmie`, so conditions 1 and 2 would flag it — but its zipcode is 98068 while Snoqualmie's is
98065, so condition 3 refuses the change. A real place survives.

In [5]:
import difflib

TRUST_THRESHOLD = 10      # a name seen 10+ times is treated as real
SIMILARITY = 0.75         # how close a rare name must be to a trusted one

city = df["city"].str.strip()
counts = city.value_counts()

trusted = set(counts[counts >= TRUST_THRESHOLD].index)
rare    = set(counts[counts <  TRUST_THRESHOLD].index)

print(f"Trusted city names : {len(trusted)}")
print(f"Rare city names    : {len(rare)}")
print()
print("Rare names (a mix of genuine small towns and typos):")
print(sorted(rare))

Trusted city names : 33
Rare city names    : 29

Rare names (a mix of genuine small towns and typos):
['Algona', 'Auburnt', 'Beaux Arts Village', 'Belleview', 'Bellvue', 'Black Diamond', 'Coronation', 'Inglewood-Finn Hill', 'Issaguah', 'Kirklund', 'Milton', 'Pacific', 'Preston', 'Ravensdale', 'Redmonde', 'Redmund', 'Samamish', 'Seaattle', 'Seatle', 'Skykomish', 'Snogualmie', 'Snoqualmie Pass', 'Sureline', 'Woodenville', 'Yarrow Point', 'auburn', 'redmond', 'sammamish', 'seattle']


In [6]:
# Majority city per zipcode, computed from TRUSTED names only so a typo can never win
zip_majority = city[city.isin(trusted)].groupby(df["zipcode"]).agg(lambda s: s.mode().iat[0])

corrections = []
new_city = city.copy()

for i in df.index[city.isin(rare)]:
    name = city[i]
    majority = zip_majority.get(df.loc[i, "zipcode"])
    if majority is None:
        continue                                  # no trusted reference for this zipcode

    case_only  = name.lower() == majority.lower()
    similarity = difflib.SequenceMatcher(None, name.lower(), majority.lower()).ratio()

    if case_only or similarity >= SIMILARITY:
        new_city[i] = majority
        corrections.append({"original": name, "corrected": majority,
                            "zipcode": df.loc[i, "zipcode"],
                            "similarity": round(similarity, 3),
                            "case_only": case_only})

corrections = pd.DataFrame(corrections)
print("Rows corrected:", len(corrections))
corrections.drop_duplicates(subset=["original", "corrected"])

Rows corrected: 20


,original,corrected,zipcode,similarity,case_only
0,seattle,Seattle,98103,1.00,True
1,Woodenville,Woodinville,98077,0.91,False
2,Seatle,Seattle,98117,0.92,False
3,Seaattle,Seattle,98108,0.93,False
4,Redmund,Redmond,98052,0.86,False
5,Belleview,Bellevue,98008,0.82,False
6,Bellvue,Bellevue,98008,0.93,False
7,Samamish,Sammamish,98074,0.94,False
8,Sureline,Shoreline,98155,0.82,False
9,Kirklund,Kirkland,98034,0.88,False


In [7]:
print(corrections.drop_duplicates(subset=["original", "corrected"]).to_string(index=False))

   original   corrected zipcode  similarity  case_only
    seattle     Seattle   98103        1.00       True
Woodenville Woodinville   98077        0.91      False
     Seatle     Seattle   98117        0.92      False
   Seaattle     Seattle   98108        0.93      False
    Redmund     Redmond   98052        0.86      False
  Belleview    Bellevue   98008        0.82      False
    Bellvue    Bellevue   98008        0.93      False
   Samamish   Sammamish   98074        0.94      False
   Sureline   Shoreline   98155        0.82      False
   Kirklund    Kirkland   98034        0.88      False
    Auburnt      Auburn   98001        0.92      False
 Snogualmie  Snoqualmie   98065        0.90      False
   Issaguah    Issaquah   98029        0.88      False
   Redmonde     Redmond   98052        0.93      False
 Coronation   Carnation   98014        0.84      False
     auburn      Auburn   98092        1.00       True
    redmond     Redmond   98052        1.00       True
  sammamis

In [8]:
n = int((new_city != df["city"]).sum())
print(f"Unique cities: {df['city'].nunique()}  ->  {new_city.nunique()}")
print()
print("Rare names deliberately LEFT UNCHANGED (real small towns):")
print(sorted(set(new_city[new_city.isin(rare)])))

df["city"] = new_city
log("Fix 2", "Corrected misspelled city names", n,
    "Rare + near-identical spelling + confirmed by zipcode majority")

Unique cities: 62  ->  44

Rare names deliberately LEFT UNCHANGED (real small towns):
['Algona', 'Beaux Arts Village', 'Black Diamond', 'Inglewood-Finn Hill', 'Milton', 'Pacific', 'Preston', 'Ravensdale', 'Skykomish', 'Snoqualmie Pass', 'Yarrow Point']
[Fix 2] Corrected misspelled city names  ->  20 rows affected, 4600 remaining


Twenty corrections, and the city count drops from 62 to 44 — which happens to match the number of
cities in `data.csv`, an independent confirmation that we found exactly the injected typos and
nothing more.

Meanwhile *Yarrow Point*, *Snoqualmie Pass*, *Preston*, *Milton*, *Beaux Arts Village*,
*Skykomish* and the other genuine small towns are untouched.

> **Why this matters more than it looks:** on Day 6 you will encode `city` as a model feature.
> Under-cleaning leaves `Seatle` as a phantom one-row category. Over-cleaning erases Yarrow Point,
> one of the most expensive areas in the dataset. Both damage the model — in opposite directions.
>
> Also note this code contains no hard-coded typo list. Hand it a new file with different
> misspellings and it still works. That difference is worth pointing out in your defense.

---
## 4. Fix 3 — the two broken dates

| Raw value | Problem | Decision |
|---|---|---|
| `20140631T000000` | June 31st does not exist | Set to **2014-06-30**, the nearest real date |
| `23052014T000000` | Written day-month-year | Reorder to **2014-05-23** |

Both fall inside the dataset's actual window (2 May – 10 July 2014), which supports the
interpretation. We repair rather than delete — these are otherwise perfectly good sales, and
`date` barely matters in this project anyway.

In [9]:
broken = df[df["date"].isna()]
print("Rows with a broken date:", len(broken))
print(broken[["date_raw", "price", "city"]].to_string(index=False))

Rows with a broken date: 2
       date_raw      price     city
20140631T000000 248,000.00 Bellevue
23052014T000000 505,000.00  Seattle


In [10]:
repairs = {
    "20140631T000000": "2014-06-30",   # June has 30 days
    "23052014T000000": "2014-05-23",   # DD-MM-YYYY written by mistake
}

for raw, corrected in repairs.items():
    mask = df["date_raw"] == raw
    df.loc[mask, "date"] = pd.Timestamp(corrected)

print("Remaining unparseable dates:", int(df["date"].isna().sum()))
log("Fix 3", "Repaired impossible / wrongly-ordered dates", len(repairs),
    "Both fall inside the real sales window; the rows are otherwise valid")

Remaining unparseable dates: 0
[Fix 3] Repaired impossible / wrongly-ordered dates  ->  2 rows affected, 4600 remaining


---
## 5. Fix 4 — `sqft_living` inconsistencies

By definition: **living area = above-ground area + basement area.**

Two rows break that identity. Because `sqft_above` and `sqft_basement` are the components and
`sqft_living` is the sum, the sum is the value we trust least — so we recompute it.

In [11]:
bad = df[df["sqft_living"] != df["sqft_above"] + df["sqft_basement"]]
print("Rows failing the identity:", len(bad))
bad[["sqft_living", "sqft_above", "sqft_basement", "price", "street", "city"]]

Rows failing the identity: 2


,sqft_living,sqft_above,sqft_basement,price,street,city
4337,1280,1280,1420,"670,000.00",746 Boylston Ave E,Seattle
4338,890,590,0,"202,000.00",701-711 26th Ave,Seattle


In [12]:
mask = df["sqft_living"] != df["sqft_above"] + df["sqft_basement"]
n = int(mask.sum())

df.loc[mask, "sqft_living"] = df.loc[mask, "sqft_above"] + df.loc[mask, "sqft_basement"]

assert (df["sqft_living"] == df["sqft_above"] + df["sqft_basement"]).all()
print("Identity now holds for all rows ✅")
log("Fix 4", "Recomputed sqft_living as above + basement", n,
    "sqft_living is a derived total; its two components are the primary measurements")

Identity now holds for all rows ✅
[Fix 4] Recomputed sqft_living as above + basement  ->  2 rows affected, 4600 remaining


---
## 6. Fix 5 — renovation year

Two separate problems live in this one column.

**Problem A — impossible values.** Some houses are recorded as renovated *before* they were built.

In [13]:
impossible = df[df["yr_renovated"].notna() & (df["yr_renovated"] < df["yr_built"])]
print("Renovated before it was built:", len(impossible))
impossible[["yr_built", "yr_renovated", "price", "city"]]

Renovated before it was built: 4


,yr_built,yr_renovated,price,city
4339,1966,"1,913.00","440,000.00",Bellevue
4340,2004,"2,003.00","690,000.00",Redmond
4341,2012,"1,912.00","375,000.00",Kent
4344,2013,"1,923.00","850,000.00",Shoreline


In [14]:
mask = df["yr_renovated"].notna() & (df["yr_renovated"] < df["yr_built"])
n = int(mask.sum())

# We know the value is wrong, but not what it should be. "Unknown" is honest; a guess is not.
df.loc[mask, "yr_renovated"] = np.nan

log("Fix 5a", "Set impossible renovation years to unknown", n,
    "A renovation cannot precede construction; the true year is unrecoverable")

[Fix 5a] Set impossible renovation years to unknown  ->  4 rows affected, 4600 remaining


**Problem B — 95% are missing.** Almost every house has a null `yr_renovated`.

That is not really missing data. It means **"this house was never renovated"** — an absence, not
an unknown. So we split the column into the two pieces of information it actually contains:

| New column | Meaning |
|---|---|
| `was_renovated` | 1 if the house was ever renovated, else 0 |
| `effective_year` | the later of `yr_built` and `yr_renovated` — how *modern* the house really is |

`effective_year` is the more useful feature: a 1920 house renovated in 2010 behaves in the market
much more like a 2010 house than a 1920 one.

In [15]:
df["was_renovated"]  = df["yr_renovated"].notna().astype(int)
df["effective_year"] = df[["yr_built", "yr_renovated"]].max(axis=1).astype(int)
df["house_age"]      = 2014 - df["yr_built"]
df["age_effective"]  = 2014 - df["effective_year"]

print(df["was_renovated"].value_counts().rename({0: "never renovated", 1: "renovated"}))
print()
df[df["was_renovated"] == 1][["yr_built", "yr_renovated", "effective_year", "age_effective"]].head()

was_renovated
never renovated    4375
renovated           225
Name: count, dtype: int64



,yr_built,yr_renovated,effective_year,age_effective
5,1938,"1,994.00",1994,20
74,1928,"1,954.00",1954,60
76,1982,"2,011.00",2011,3
96,1909,"1,998.00",1998,16
104,1942,"1,958.00",1958,56


In [16]:
log("Fix 5b", "Created was_renovated / effective_year / house_age", len(df),
    "Null meant 'never renovated', not 'unknown'; effective_year captures true modernity")

[Fix 5b] Created was_renovated / effective_year / house_age  ->  4600 rows affected, 4600 remaining


---
## 7. Fix 6 — houses with 0 bedrooms and 0 bathrooms

Look at these two before deciding anything.

In [17]:
zero_rooms = df[(df["bedrooms"] == 0) | (df["bathrooms"] == 0)]
print("Rows with 0 bedrooms or 0 bathrooms:", len(zero_rooms))
zero_rooms[["price", "bedrooms", "bathrooms", "sqft_living", "floors", "yr_built", "city"]]

Rows with 0 bedrooms or 0 bathrooms: 2


,price,bedrooms,bathrooms,sqft_living,floors,yr_built,city
2365,"1,095,000.00",0.00,0.00,3064,3.50,1990,Seattle
3209,"1,295,648.00",0.00,0.00,4810,2.00,1990,Redmond


These are **not** empty plots of land. They are a 3,064 sqft house in Seattle sold for $1.09M and
a 4,810 sqft house in Redmond sold for $1.30M. A million-dollar house with zero bathrooms does not
exist — the values were simply never entered.

`bedrooms` and `bathrooms` are **features**, not the target, so imputation is legitimate here.
We estimate from houses of a similar size (within ±10% of the same living area).

In [18]:
reference = df[(df["bedrooms"] > 0) & (df["bathrooms"] > 0) & (df["price"] > 0)]

def impute_rooms(row):
    similar = reference[reference["sqft_living"].between(row["sqft_living"] * 0.9,
                                                         row["sqft_living"] * 1.1)]
    return pd.Series({
        "bedrooms":  similar["bedrooms"].median(),
        "bathrooms": similar["bathrooms"].median(),
        "n_similar": len(similar),
    })

targets = df.index[(df["bedrooms"] == 0) | (df["bathrooms"] == 0)]

for i in targets:
    est = impute_rooms(df.loc[i])
    print(f"row {i}: {df.loc[i, 'sqft_living']:>5.0f} sqft  ->  "
          f"{est['bedrooms']:.0f} bed / {est['bathrooms']:.2f} bath  "
          f"(from {est['n_similar']:.0f} similar houses)")
    df.loc[i, "bedrooms"]  = est["bedrooms"]
    df.loc[i, "bathrooms"] = est["bathrooms"]

log("Fix 6", "Imputed bedrooms/bathrooms from similarly-sized houses", len(targets),
    "Features may be imputed; both houses are large, expensive and otherwise complete")

row 2365:  3064 sqft  ->  4 bed / 2.50 bath  (from 498 similar houses)
row 3209:  4810 sqft  ->  4 bed / 3.50 bath  (from 86 similar houses)
[Fix 6] Imputed bedrooms/bathrooms from similarly-sized houses  ->  2 rows affected, 4600 remaining


> **Be ready for this question:** *"Why impute here but delete the missing prices?"*
> Because `bedrooms` is an input and `price` is the answer. Guessing an input adds a little noise.
> Guessing the answer means grading the model against your own guess.

---
## 8. Fix 7 — the 248 missing prices

The decisive step of the whole notebook.

In [19]:
missing_price = df[df["price"] == 0]
print("Houses with price = 0:", len(missing_price))
print(f"That is {len(missing_price) / len(df) * 100:.1f}% of the dataset")
print()
print("Are these rows broken in other ways too?")
missing_price[["bedrooms", "bathrooms", "sqft_living", "yr_built", "condition"]].describe().T[["count", "mean", "min", "max"]]

Houses with price = 0: 248
That is 5.4% of the dataset

Are these rows broken in other ways too?


,count,mean,min,max
bedrooms,248.00,3.45,1.00,6.00
bathrooms,248.00,2.19,1.00,6.25
sqft_living,248.00,"2,201.72",520.00,"8,020.00"
yr_built,248.00,"1,969.66","1,901.00","2,014.00"
condition,248.00,3.52,2.00,5.00


Every other column is perfectly normal. Only the price is absent.

### Why we delete rather than impute

1. `price` is the **target**. A model trained on imputed prices learns the imputation rule, not the
   housing market — and its error metrics become self-referential and meaningless.
2. There is no way to recover the true value. Any fill is an invention.
3. 248 rows is **5.4%** of the data. We keep 94.6% of a clean signal instead of 100% of a
   partly-fabricated one. That is a good trade.

> This is precisely what `data.csv` got wrong: 199 of these were filled with group averages,
> which is why prices like `$237,227.857143` appear in it.

In [20]:
before = len(df)
df = df[df["price"] > 0].reset_index(drop=True)

log("Fix 7", "Removed rows with price = 0", before - len(df),
    "Missing TARGET variable; imputing it would fabricate ground truth")

[Fix 7] Removed rows with price = 0  ->  248 rows affected, 4352 remaining


---
## 9. Fix 8 — impossible prices

Some remaining prices are present but implausible. The trap here is that **"expensive" and
"wrong" are not the same thing.** A $7M mansion in Bellevue is real. We must not delete genuine
luxury housing just because it is extreme.

### A better detector than raw price: price per square foot, relative to the neighbourhood

A large house *should* cost more. What no house can do is cost **145 times the going rate on its
own street**. So we compare each house to the median price per sqft of **its own zipcode**.

In [21]:
df["price_per_sqft"] = df["price"] / df["sqft_living"]
zip_median = df.groupby("zipcode")["price_per_sqft"].transform("median")
df["pps_ratio"] = df["price_per_sqft"] / zip_median

print("Distribution of pps_ratio (1.0 = exactly the neighbourhood median):")
print(df["pps_ratio"].describe([.01, .25, .5, .75, .99]))

Distribution of pps_ratio (1.0 = exactly the neighbourhood median):


count   4,352.00
mean        1.08
std         2.22
min         0.06
1%          0.56
25%         0.88
50%         1.00
75%         1.15
99%         1.92
max       144.94
Name: pps_ratio, dtype: float64


In [22]:
LOW, HIGH = 0.25, 4.0   # a house priced under 1/4 or over 4x its neighbourhood rate

flagged = df[(df["pps_ratio"] < LOW) | (df["pps_ratio"] > HIGH)]
print(f"Flagged for review: {len(flagged)} rows")
print()
flagged[["price", "sqft_living", "price_per_sqft", "pps_ratio", "city"]].sort_values("pps_ratio")

Flagged for review: 7 rows

,price,sqft_living,price_per_sqft,pps_ratio,city
4351,"7,800.00",780,10.00,0.06,Tukwila
4345,"84,350.00",2630,32.07,0.07,Yarrow Point
4349,"188,000.00",3260,57.67,0.10,Medina
4347,"2,110,000.00",2100,"1,004.76",6.12,Tukwila
4348,"2,199,900.00",1120,"1,964.20",12.55,Covington
4346,"12,899,000.00",2190,"5,889.95",16.87,Seattle
4350,"26,590,000.00",1180,"22,533.90",144.94,Kent


### Check that the threshold is not arbitrary

A threshold you cannot defend is a threshold an examiner will attack. So let us look at what sits
*just inside* the boundary and confirm there is a real gap, rather than a smooth continuum that we
sliced at a convenient point.

In [23]:
ordered = df["pps_ratio"].sort_values()

print("Lowest 10 ratios:")
print(ordered.head(10).round(3).to_string())
print()
print("Highest 10 ratios:")
print(ordered.tail(10).round(3).to_string())

Lowest 10 ratios:
4351   0.06
4345   0.07
4349   0.10
1042   0.27
1028   0.35
1728   0.36
1236   0.39
2194   0.40
3635   0.42
223    0.43

Highest 10 ratios:
960      2.76
1238     3.09
2416     3.17
404      3.22
1191     3.68
2919     3.93
4347     6.12
4348    12.55
4346    16.87
4350   144.94


There is a clear **gap**, not a gradient:

- On the low side: the flagged rows sit at **0.06 – 0.10**, while the most extreme legitimate
  house sits at **0.27**.
- On the high side: the flagged rows sit at **6.1 – 145**, while the most extreme legitimate
  house sits at **3.9**.

Nothing lives in between. The cut is not a tuning choice — it separates two genuinely different
populations.

### And they fail the common-sense test individually

| Price | Size | Location | Why it is impossible |
|---|---|---|---|
| $26,590,000 | 1,180 sqft | Kent | $22,534/sqft in a modest suburb — 145× the local rate |
| $12,899,000 | 2,190 sqft | Seattle | $5,890/sqft; the real Seattle median is ~$300 |
| $2,199,900 | 1,120 sqft | Covington | A small house in a low-cost suburb |
| $2,110,000 | 2,100 sqft | Tukwila | Same problem |
| $188,000 | 3,260 sqft | Medina | Medina is one of the wealthiest towns in the USA |
| $84,350 | 2,630 sqft | Yarrow Point | Likewise — a waterfront luxury enclave |
| $7,800 | 780 sqft | Tukwila | $10/sqft is not a housing transaction |

**Keep in mind what we are *not* deleting:** the $7,062,500 / 10,040 sqft house in Bellevue stays.
It is expensive because it is enormous and in a wealthy area — its ratio is 1.0, dead on its
neighbourhood's rate. Removing it would be throwing away real market information.

The remaining natural skew is handled later by modelling `log(price)`, not by deleting rows.

In [24]:
before = len(df)
df = df[(df["pps_ratio"] >= LOW) & (df["pps_ratio"] <= HIGH)].reset_index(drop=True)

log("Fix 8", "Removed impossible prices (zipcode-relative price/sqft)", before - len(df),
    "Priced <0.25x or >4x their own neighbourhood rate; a clear gap separates them from real luxury")

[Fix 8] Removed impossible prices (zipcode-relative price/sqft)  ->  7 rows affected, 4345 remaining


---
## 10. Drop columns that carry no information

In [25]:
drop_cols = {
    "country":    "single value 'USA' for every row",
    "address":    "already split into street / city / statezip / zipcode",
    "statezip":   "state is constant (WA); zipcode extracted separately",
    "date_raw":   "working column used only for date repair",
    "street":     "nearly unique per row; unusable as a feature",
    "pps_ratio":  "diagnostic used for outlier detection only",
}

for col, reason in drop_cols.items():
    if col in df.columns:
        print(f"  drop {col:15s} — {reason}")

df = df.drop(columns=[c for c in drop_cols if c in df.columns])
log("Fix 9", "Dropped uninformative / redundant columns", len(drop_cols),
    "Constant, redundant, or too granular to generalise")

  drop country         — single value 'USA' for every row
  drop address         — already split into street / city / statezip / zipcode
  drop statezip        — state is constant (WA); zipcode extracted separately
  drop date_raw        — working column used only for date repair
  drop street          — nearly unique per row; unusable as a feature
  drop pps_ratio       — diagnostic used for outlier detection only
[Fix 9] Dropped uninformative / redundant columns  ->  6 rows affected, 4345 remaining


> **`price_per_sqft` stays**, but treat it with care. It is derived *from* the target, so feeding
> it into the price model would be **target leakage** — the model would effectively be handed the
> answer. Keep it for the Day 4/5 business analysis, and exclude it from the regression features
> on Day 6. Write this warning into your notebook so you remember.

---
## 11. Final validation

Re-run every Day 1 check against the cleaned data. All of them must now pass.

In [26]:
checks = {
    "price = 0":                     int((df["price"] == 0).sum()),
    "bedrooms = 0":                  int((df["bedrooms"] == 0).sum()),
    "bathrooms = 0":                 int((df["bathrooms"] == 0).sum()),
    "duplicate rows":                int(df.duplicated().sum()),
    "unparseable dates":             int(df["date"].isna().sum()),
    "sqft_living != above+basement": int((df["sqft_living"] != df["sqft_above"] + df["sqft_basement"]).sum()),
    "renovated before built":        int((df["yr_renovated"].notna() & (df["yr_renovated"] < df["yr_built"])).sum()),
    "missing values (excl. yr_renovated)": int(df.drop(columns=["yr_renovated"]).isna().sum().sum()),
}

for name, value in checks.items():
    print(f"{'PASS ✅' if value == 0 else 'FAIL ❌'}  {name:38s} {value}")

assert all(v == 0 for v in checks.values()), "Some checks did not pass"
print()
print("All quality checks passed.")

PASS ✅  price = 0                              0
PASS ✅  bedrooms = 0                           0
PASS ✅  bathrooms = 0                          0
PASS ✅  duplicate rows                         0
PASS ✅  unparseable dates                      0
PASS ✅  sqft_living != above+basement          0
PASS ✅  renovated before built                 0
PASS ✅  missing values (excl. yr_renovated)    0

All quality checks passed.


In [27]:
print("Cities:", df["city"].nunique(), " Zipcodes:", df["zipcode"].nunique())
print()
df[["price", "bedrooms", "bathrooms", "sqft_living", "house_age", "price_per_sqft"]].describe().T[["count", "mean", "min", "50%", "max"]]

Cities: 44  Zipcodes: 77



,count,mean,min,50%,max
price,"4,345.00","555,721.12","80,000.00","471,000.00","7,062,500.00"
bedrooms,"4,345.00",3.40,1.00,3.00,9.00
bathrooms,"4,345.00",2.16,0.75,2.25,8.00
sqft_living,"4,345.00","2,136.18",370.00,"1,970.00","13,540.00"
house_age,"4,345.00",43.14,0.00,38.00,114.00
price_per_sqft,"4,345.00",264.64,87.65,247.66,800.00


Compare this to Day 1:

| | Before | After |
|---|---|---|
| Max price | $26,590,000 | ~$7,060,000 |
| Min price | $0 | ~$80,000 |
| Mean price | $534,524 | lower, and no longer dragged by phantom values |

The maximum is now a real Bellevue mansion instead of a data-entry error. The remaining skew is
genuine market structure, and we will handle it with a log transform — not with deletion.

---
## 12. Save and report

In [28]:
out = DATA / "data_clean.csv"
df.to_csv(out, index=False)

print("Saved:", out.resolve())
print("Final shape:", df.shape)
print()
print("Columns:")
print(list(df.columns))

Saved: C:\Users\MY LAP\Documents\Graduation Project\Data\data_clean.csv
Final shape: (4345, 21)

Columns:
['date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'city', 'zipcode', 'was_renovated', 'effective_year', 'house_age', 'age_effective', 'price_per_sqft']


In [29]:
log_df = pd.DataFrame(cleaning_log)
print(f"Started with {start_rows:,} rows  ->  finished with {len(df):,} rows")
print(f"Removed {start_rows - len(df):,} rows ({(start_rows - len(df)) / start_rows * 100:.1f}%)")
print()
log_df

Started with 4,601 rows  ->  finished with 4,345 rows
Removed 256 rows (5.6%)



,step,action,rows,reason,rows_remaining
0,Fix 1,Removed exact duplicate rows,1,Identical in every column; a single sale recor...,4600
1,Fix 2,Corrected misspelled city names,20,Rare + near-identical spelling + confirmed by ...,4600
2,Fix 3,Repaired impossible / wrongly-ordered dates,2,Both fall inside the real sales window; the ro...,4600
3,Fix 4,Recomputed sqft_living as above + basement,2,sqft_living is a derived total; its two compon...,4600
4,Fix 5a,Set impossible renovation years to unknown,4,A renovation cannot precede construction; the ...,4600
5,Fix 5b,Created was_renovated / effective_year / house...,4600,"Null meant 'never renovated', not 'unknown'; e...",4600
6,Fix 6,Imputed bedrooms/bathrooms from similarly-size...,2,Features may be imputed; both houses are large...,4600
7,Fix 7,Removed rows with price = 0,248,Missing TARGET variable; imputing it would fab...,4352
8,Fix 8,Removed impossible prices (zipcode-relative pr...,7,Priced <0.25x or >4x their own neighbourhood r...,4345
9,Fix 9,Dropped uninformative / redundant columns,6,"Constant, redundant, or too granular to genera...",4345


---

## Cleaning decisions — summary for your report

| # | Problem | Rows | Action | Justification |
|---|---|---|---|---|
| 1 | Exact duplicate | 1 | Removed | One sale recorded twice |
| 2 | Misspelled city names | 20 | Corrected via rarity + similarity + zipcode | Conservative rule; real small towns preserved |
| 3 | Impossible / reordered dates | 2 | Repaired | Both fall inside the real sales window |
| 4 | `sqft_living` ≠ above + basement | 2 | Recomputed | The total is derived; components are primary |
| 5a | Renovated before built | 4 | Set to unknown | Physically impossible; true value unrecoverable |
| 5b | `yr_renovated` null in 95% | 4,368 | → `was_renovated` + `effective_year` | Null meant "never renovated", not "unknown" |
| 6 | 0 bedrooms / 0 bathrooms | 2 | Imputed from similar-sized houses | **Features** may be imputed |
| 7 | `price` = 0 | 248 | **Removed** | **Target** may never be imputed |
| 8 | Impossible price per sqft | 7 | Removed | <0.25× or >4× their own neighbourhood rate |
| 9 | Uninformative columns | 6 cols | Dropped | Constant, redundant or too granular |

**Result:** 4,601 → ~4,345 rows (**5.6% removed**), and every quality check passes.

---

## ✅ Day 2 checklist

- [ ] Every cell runs, and the final assertion passes
- [ ] `data_clean.csv` saved
- [ ] Cleaning-decisions table rewritten **in your own words**
- [ ] You can justify each decision without reading from the screen
- [ ] Committed: `git add . && git commit -m "Day 2: cleaning"`

## ➡️ Day 3 preview

Distributions: price skew, the log transform, and boxplots by city. First charts saved to
`reports/figures/`.

## 🎓 Be ready to answer

1. Why delete the 248 missing prices instead of filling them?
2. Why impute bedrooms but not price? *(Feature vs target — know this cold.)*
3. Why is a $7M house kept while a $2.1M house is deleted?
4. How did you choose the 0.25 / 4.0 thresholds, and how do you know they are not arbitrary?
5. You removed 5.6% of the data — how do you know that did not bias the result?
6. Why is `price_per_sqft` dangerous to use as a model feature?
7. Why not just replace every city with whatever its zipcode says?